In [1]:
import matplotlib.pyplot as plt
import pandas as pd

import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.impute import SimpleImputer

from collections import defaultdict
from multiprocessing import Pool, cpu_count
from joblib import Parallel, delayed, load
from tqdm.notebook import tqdm

import ast
import glob
import pickle
import dask
import os
import dask.dataframe as dd
from dask.distributed import Client
client = Client(n_workers=20, memory_limit="10GB", interface='lo')

from pprint import pprint

### Load kmeans data and classifiers

In [2]:
augmented_data_path = "../../data/augmented_us-counties-states_latest.csv"
augmented_df = (dd.read_csv(augmented_data_path, assume_missing=True))
# feature data
hhs_X_path = "./hhs_kmeans_data.csv"
hhs_X_df = dd.read_csv(hhs_X_path, assume_missing=True).compute()
X = hhs_X_df.drop(["fips", "county", "state"], axis=1)
imp = SimpleImputer(strategy='mean')
X = imp.fit_transform(X)
hhs_X_df.head()
hhs_y_df = augmented_df[["fips","datetime","days_from_start","rolled_cases"]].compute().sort_values(["fips","days_from_start"])
hhs_y_df["log_rolled_cases"] = np.log(hhs_y_df["rolled_cases"] + 1.1)
hhs_y_df["shifted_log_rolled_cases"] = hhs_y_df.groupby("fips")["log_rolled_cases"].shift(-7)


/home/zwang937/anaconda3/lib/python3.7/site-packages/pandas/core/arraylike.py:364: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [3]:
kmeans_classifiers_folder_path = "./kmeans_classifiers"
def load_kmeans(fname):
    return load(fname)
kmeans_classifiers_paths = glob.glob("./kmeans_classifiers/kmeans_*.joblib")
K_list = [int(s.split("/")[2].split("_")[1].split(".")[0]) for s in kmeans_classifiers_paths]
K_list = sorted(K_list)
kmeans_classifiers = Parallel(n_jobs=-1)(delayed(load_kmeans)(f) for f in sorted(kmeans_classifiers_paths))

In [4]:
K_list

[2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 100,
 200,
 300,
 400,
 500,
 600,
 700,
 800,
 900,
 1000,
 1100,
 1200,
 1300,
 1400,
 1500,
 1600,
 1700,
 1800,
 1900,
 2000,
 2100,
 2200,
 2300,
 2400,
 2500,
 2600,
 2700,
 2800,
 2900,
 3000,
 3100]

In [5]:
(kmeans_classifiers)[-2].cluster_centers_

array([[ 3.29661751e+01, -9.30209320e+01,  3.56708625e+03, ...,
         1.21212121e-01,  0.00000000e+00,  5.34498834e+00],
       [ 2.98572730e+01, -9.53930370e+01,  1.43208000e+05, ...,
         0.00000000e+00,  0.00000000e+00,  6.00000000e+00],
       [ 3.68615796e+01, -9.06363381e+01,  3.48081618e+04, ...,
         3.67647059e-01,  0.00000000e+00,  4.92647059e+00],
       ...,
       [ 4.06485030e+01, -9.72858178e+01,  6.03917476e+03, ...,
         4.75728155e-01,  0.00000000e+00,  5.33009709e+00],
       [ 4.17932888e+01, -1.02874002e+02,  1.64553465e+03, ...,
         2.95049505e-01,  0.00000000e+00,  7.14455446e+00],
       [ 3.41963980e+01, -1.18261862e+02,  3.57178000e+05, ...,
         1.00000000e+00,  0.00000000e+00,  9.00000000e+00]])

In [6]:
def predict_label(kmeans_classifier):
    k_clusters = kmeans_classifier.get_params()["n_clusters"]
    labels = kmeans_classifier.predict(X).compute()
    label_col = pd.DataFrame({"kmeans_k={}_labels".format(k_clusters): labels})
    return label_col
labels_arr = Parallel(n_jobs=-1)(delayed(predict_label)(c) for c in (kmeans_classifiers))

In [7]:
hhs_X_w_clusters_df = pd.concat([hhs_X_df, pd.concat(labels_arr, axis=1)], axis=1)
hhs_X_w_clusters_df.to_csv("./hhs_X_w_clusters.csv", index=False)

In [8]:
hhs_labels_df = hhs_X_w_clusters_df[["fips", "county","state"] + ["kmeans_k={}_labels".format(k) for k in K_list]]
hhs_labels_df.head()

,fips,county,state,kmeans_k=2_labels,kmeans_k=3_labels,kmeans_k=4_labels,kmeans_k=5_labels,kmeans_k=6_labels,kmeans_k=7_labels,kmeans_k=8_labels,...,kmeans_k=2200_labels,kmeans_k=2300_labels,kmeans_k=2400_labels,kmeans_k=2500_labels,kmeans_k=2600_labels,kmeans_k=2700_labels,kmeans_k=2800_labels,kmeans_k=2900_labels,kmeans_k=3000_labels,kmeans_k=3100_labels
0,1001.0,Autauga,Alabama,0,0,0,0,0,0,5,...,1513,1513,1513,1513,1513,1513,1513,1513,1121,1121
1,1003.0,Baldwin,Alabama,0,0,0,0,0,0,0,...,372,372,372,372,372,372,372,372,488,488
2,1005.0,Barbour,Alabama,0,0,0,0,0,0,5,...,1763,1763,1763,1763,1763,1763,1763,1763,1063,1063
3,1007.0,Bibb,Alabama,0,0,0,0,0,0,5,...,1759,1759,1759,1759,1759,1759,1759,1759,1362,1362
4,1009.0,Blount,Alabama,0,0,0,0,0,0,5,...,312,312,312,312,312,312,312,312,1575,1575


In [9]:
n_fips = hhs_labels_df.shape[0]
hhs_labels_df["kmeans_k={}_labels".format(n_fips)] = hhs_labels_df.index
hhs_labels_df["kmeans_k={}_labels".format(1)] = 0
hhs_labels_df

/home/zwang937/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  
/home/zwang937/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  This is separate from the ipykernel package so we can avoid doing imports until


,fips,county,state,kmeans_k=2_labels,kmeans_k=3_labels,kmeans_k=4_labels,kmeans_k=5_labels,kmeans_k=6_labels,kmeans_k=7_labels,kmeans_k=8_labels,...,kmeans_k=2400_labels,kmeans_k=2500_labels,kmeans_k=2600_labels,kmeans_k=2700_labels,kmeans_k=2800_labels,kmeans_k=2900_labels,kmeans_k=3000_labels,kmeans_k=3100_labels,kmeans_k=3136_labels,kmeans_k=1_labels
0,1001.0,Autauga,Alabama,0,0,0,0,0,0,5,...,1513,1513,1513,1513,1513,1513,1121,1121,0,0
1,1003.0,Baldwin,Alabama,0,0,0,0,0,0,0,...,372,372,372,372,372,372,488,488,1,0
2,1005.0,Barbour,Alabama,0,0,0,0,0,0,5,...,1763,1763,1763,1763,1763,1763,1063,1063,2,0
3,1007.0,Bibb,Alabama,0,0,0,0,0,0,5,...,1759,1759,1759,1759,1759,1759,1362,1362,3,0
4,1009.0,Blount,Alabama,0,0,0,0,0,0,5,...,312,312,312,312,312,312,1575,1575,4,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3131,56039.0,Teton,Wyoming,0,0,0,4,4,4,7,...,482,482,482,482,482,482,434,434,3131,0
3132,56041.0,Uinta,Wyoming,0,0,0,4,4,4,7,...,5,5,5,5,5,5,275,275,3132,0
3133,56043.0,Washakie,Wyoming,0,0,0,4,4,4,7,...,1675,1675,1675,1675,2748,2748,2756,2756,3133,0
3134,56045.0,Weston,Wyoming,0,0,0,4,4,4,7,...,2043,2043,2043,2043,2043,2043,949,949,3134,0


In [10]:
hhs_clustered_panel_data = pd.merge(hhs_y_df, hhs_labels_df, how="inner", on="fips")
hhs_clustered_panel_data["log_rolled_cases"] = hhs_clustered_panel_data["log_rolled_cases"].apply(lambda x: max(0,x))
print(hhs_clustered_panel_data.shape)
(hhs_clustered_panel_data.head())
hhs_clustered_panel_data.to_csv("./hhs_clustered_panel_data.csv", index=False)

(3384567, 60)


### Verify that all clusters have been assigned

In [11]:
for K in K_list:
    n_clusters = len(hhs_clustered_panel_data["kmeans_k={}_labels".format(K)].unique())
    if K != n_clusters:
        print("K={}, number of assigned clusters is {}".format(K, n_clusters))


In [12]:
len(hhs_clustered_panel_data["kmeans_k=3000_labels"].unique())

3000

In [13]:
hhs_clustered_panel_data

,fips,datetime,days_from_start,rolled_cases,log_rolled_cases,shifted_log_rolled_cases,county,state,kmeans_k=2_labels,kmeans_k=3_labels,...,kmeans_k=2400_labels,kmeans_k=2500_labels,kmeans_k=2600_labels,kmeans_k=2700_labels,kmeans_k=2800_labels,kmeans_k=2900_labels,kmeans_k=3000_labels,kmeans_k=3100_labels,kmeans_k=3136_labels,kmeans_k=1_labels
0,1001.0,2020-03-30,69.0,5.142857,1.831438,2.469309,Autauga,Alabama,0,0,...,1513,1513,1513,1513,1513,1513,1121,1121,0,0
1,1001.0,2020-03-31,70.0,6.000000,1.960095,2.528012,Autauga,Alabama,0,0,...,1513,1513,1513,1513,1513,1513,1121,1121,0,0
2,1001.0,2020-04-01,71.0,6.857143,2.074070,2.550561,Autauga,Alabama,0,0,...,1513,1513,1513,1513,1513,1513,1121,1121,0,0
3,1001.0,2020-04-02,72.0,7.428571,2.143422,2.625703,Autauga,Alabama,0,0,...,1513,1513,1513,1513,1513,1513,1121,1121,0,0
4,1001.0,2020-04-03,73.0,8.285714,2.239189,2.676117,Autauga,Alabama,0,0,...,1513,1513,1513,1513,1513,1513,1121,1121,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3384562,99999.0,2023-03-19,1153.0,13972.285714,9.544910,NaN,New York City,New York,0,2,...,13,13,13,13,13,13,15,15,3135,0
3384563,99999.0,2023-03-20,1154.0,13317.571429,9.496922,NaN,New York City,New York,0,2,...,13,13,13,13,13,13,15,15,3135,0
3384564,99999.0,2023-03-21,1155.0,12458.714286,9.430264,NaN,New York City,New York,0,2,...,13,13,13,13,13,13,15,15,3135,0
3384565,99999.0,2023-03-22,1156.0,12154.857143,9.405575,NaN,New York City,New York,0,2,...,13,13,13,13,13,13,15,15,3135,0


In [14]:
grouped = hhs_clustered_panel_data.groupby('fips')

for fips, group in grouped:
    missing_days = group['days_from_start'].diff().gt(1).sum()
    if missing_days > 0:
        print(f"Gap(s) found in 'days_from_start' for fips {fips}: {missing_days} gap(s)")